# Bike Rentals: Data Exploration & Preprocessing

## 1. Objective

---

The company runs a city-wide bike sharing service. Customers either book a bike in advance or pick one up directly. To plan daily bike allocation, it is starting a machine learning project on its historical data.

This notebook does not build a model. First it checks the four raw CSV sources — their structure, quality, and time coverage. Then it builds the combined hourly dataset that the Dagster preprocessing pipeline will produce.

- registered bike rentals — booking events
- direct pickup bike rentals — walk-up rentals
- weather data — hourly conditions and measurements
- holiday calendar — public holidays

The findings guide a reproducible preprocessing pipeline in Dagster.

## 2. Imports & Setup

---

The analysis uses pandas for data handling, numpy for missing-value handling, and pathlib for file paths.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Path to the raw data, relative to this notebook (bike-rental/notebooks/).
DATA_RAW_BIKE_RENTAL = Path("../data/raw")

registered_rentals_file_name = "registered_bike_rentals.csv"
direct_rentals_file_name = "direct_pickup_bike_rentals.csv"
weather_file_name = "weather.csv"
holiday_calendar_file_name = "holidays.csv"

## 3. Load Datasets

---

Load the four raw CSV files into separate DataFrames. No parsing or type casting yet — that work belongs to the preprocessing pipeline.

In [3]:
registered_rentals_data = pd.read_csv(DATA_RAW_BIKE_RENTAL / registered_rentals_file_name)
direct_rentals_data    = pd.read_csv(DATA_RAW_BIKE_RENTAL / direct_rentals_file_name)
weather_data           = pd.read_csv(DATA_RAW_BIKE_RENTAL / weather_file_name)
holiday_calendar_data  = pd.read_csv(DATA_RAW_BIKE_RENTAL / holiday_calendar_file_name)

## 4. First Data Audit

---

Before going deeper, we check each dataset for basic problems. Wrong types, missing values, or duplicate rows can break later steps without warning.

The same first check runs on every source:

- size — row and column counts
- types — do the dtypes match the data?
- missing values — find gaps and how big they are
- duplicate rows — exact repeats that can distort counts
- example rows — check real values against expectations

These results set the baseline for the per-dataset checks below.

In [4]:
def basic_audit_dataframe(df, name, date_cols=None, categorical_cols=None):
    """Print structural audit of a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Data to audit.
    name : str
        Label for the printout header.
    date_cols : list[str], optional
        Columns to parse as datetime and report min..max range.
    categorical_cols : list[str], optional
        Columns to show value_counts for.
    """
    print(f"Dataset:  {name}\n")
    df.info()

    print("\nDescribe:")
    print(df.describe(include="all").to_string())

    print(f"\nUnique counts:\n{df.nunique().to_string()}")

    for col in date_cols or []:
        ts = pd.to_datetime(df[col], format="ISO8601")
        print(f"{col} range: {ts.min()} → {ts.max()}")

    print("\nFirst 5 rows:")
    print(df.head().to_string())

    for col in categorical_cols or []:
        print(f"\nvalue_counts of [{col}]:")
        print(df[col].value_counts().to_string())


### 4.1 Registered Bike Rentals

In [5]:
basic_audit_dataframe(
    registered_rentals_data,
    registered_rentals_file_name,
    date_cols=["datetime"],
)

Dataset:  registered_bike_rentals.csv

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype
---  ------       -----
 0   id           int64
 1   datetime     str  
 2   user_id      int64
 3   location_id  int64
dtypes: int64(3), str(1)
memory usage: 81.6 MB

Describe:
                  id             datetime       user_id   location_id
count   2.672662e+06              2672662  2.672662e+06  2.672662e+06
unique           NaN              2564503           NaN           NaN
top              NaN  2012-05-01 17:25:19           NaN           NaN
freq             NaN                    5           NaN           NaN
mean    1.336332e+06                  NaN  1.500238e+02  9.994752e+00
std     7.715312e+05                  NaN  8.690642e+01  6.056529e+00
min     1.000000e+00                  NaN  0.000000e+00  0.000000e+00
25%     6.681662e+05                  NaN  7.500000e+01  5.000000e+00
50%     1.336332e+06        

### Observations

- Each row is one rental event made by a registered user.
- `datetime` is stored as a string — needs parsing before any time-based step.

**Range checks:**

- `id` is sequential and unique — the expected primary key.
- `user_id` (0..300) and `location_id` (0..20) are fully covered: every user and location is active over the 2 years. Plausible.
- `datetime` covers the full 2-year window (2011-01-01 → 2012-12-31) with no parse failures.

### 4.2 Direct pickup Bike Rentals

In [6]:
basic_audit_dataframe(
    direct_rentals_data,
    direct_rentals_file_name,
    date_cols=["datetime"],
)

Dataset:  direct_pickup_bike_rentals.csv

<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           620017 non-null  int64
 1   datetime     620017 non-null  str  
 2   user_id      620017 non-null  int64
 3   location_id  620017 non-null  int64
dtypes: int64(3), str(1)
memory usage: 18.9 MB

Describe:
                   id             datetime        user_id    location_id
count   620017.000000               620017  620017.000000  620017.000000
unique            NaN               611216            NaN            NaN
top               NaN  2012-04-21 15:44:14            NaN            NaN
freq              NaN                    4            NaN            NaN
mean    310009.000000                  NaN     150.064289       9.987275
std     178983.635264                  NaN      86.885523       6.059862
min          1.000000                  NaN  

### Observations

- Each row is one direct pickup rental (no prior booking).
- The schema is identical to the registered rentals dataset — easy to combine.

**Range checks:**

- `user_id` (0..300) and `location_id` (0..20) **match registered exactly** — both sources cover the same users and locations. A good sign the sources match.
- `datetime` covers the same 2-year window as registered (2011-01-01 → 2012-12-31) — the sources cover the same time period.
- Volume is ~4.3× lower than registered — plausible: fewer walk-ups than bookings.

### 4.3 Weather

In [7]:
basic_audit_dataframe(
    weather_data,
    weather_file_name,
    date_cols=["datetime"],
    categorical_cols=["conditions"],
)

Dataset:  weather.csv

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       17379 non-null  int64  
 1   datetime                 17379 non-null  str    
 2   conditions               17379 non-null  str    
 3   temperature_c            17379 non-null  float64
 4   perceived_temperature_c  17379 non-null  float64
 5   humidity                 17379 non-null  float64
 6   windspeed_kmh            17379 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 950.5 KB

Describe:
                id             datetime conditions  temperature_c  perceived_temperature_c      humidity  windspeed_kmh
count   17379.0000                17379      17379   17379.000000             17379.000000  17379.000000   17379.000000
unique         NaN                17379          4            NaN                  

### Observations

- Each row is one hour of weather data.
- `conditions` is a 4-level ordered category (`clear < clouds < light_rain < heavy_rain`); we encode it as an ordinal integer 0..3 in §6.3 to keep the order in one column.

**Range checks:**

- `datetime` is hourly and covers the same 2-year window as the rental sources (2011-01-01 → 2012-12-31) — the sources cover the same time period.
- `temperature_c` (−7.1..39.0 °C) and `perceived_temperature_c` (−16.0..50.0 °C) are realistic for a temperate climate. The perceived range is wider, as expected from wind chill and heat effects.
- `windspeed_kmh` (0..57) has no negative values and a realistic peak.
- **`humidity` reaches exactly `0.0%`** — real humidity almost never hits zero. Flag for §5: a real value or a broken sensor?
- **`heavy_rain` has only 3 rows** — a very rare class. No special handling needed; ordinal encoding keeps it as the highest level, and the rarity is something to handle at the modeling stage.
- Row count (17,379) is ~165 short of the 17,544 hourly slots in the 2-year window — checked against the other tables in §5.2.

### 4.4 Holiday calendar

In [8]:
basic_audit_dataframe(
    holiday_calendar_data,
    holiday_calendar_file_name,
    date_cols=["date"],
    categorical_cols=["holiday"],
)

Dataset:  holidays.csv

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       21 non-null     int64
 1   date     21 non-null     str  
 2   holiday  21 non-null     str  
dtypes: int64(1), str(2)
memory usage: 636.0 bytes

Describe:
               id        date                                 holiday
count   21.000000          21                                      21
unique        NaN          21                                      14
top           NaN  2011-01-17  Dr. Martin Luther King, Jr.'s Birthday
freq          NaN           1                                       2
mean    11.000000         NaN                                     NaN
std      6.204837         NaN                                     NaN
min      1.000000         NaN                                     NaN
25%      6.000000         NaN                                     NaN
50%     11.00

### Observations

- Each row is one public holiday; it becomes a binary `is_holiday` flag during the join.
- The dataset is small and used only as an enrichment source.

**Range checks:**

- ~10–11 holidays per year. Plausible.
- `date` range (2011-01-17 → 2012-12-25) is inside the 2-year window — consistent with the rental and weather sources.
- **`holiday` has 14 unique names vs 21 rows** — the `(observed)` suffix marks holidays moved off a weekend (e.g. `Christmas Day (observed)` in 2011 vs `Christmas Day` in 2012).

## 5. Specific Data Quality Checks

---

Section 4 ran a uniform first-pass audit on every source. This section goes deeper into the specific data quality issues flagged there: near-duplicate rental events, cross-source coverage gaps, and a sensor anomaly in the weather data. Each subsection makes a concrete decision about what to do (or explicitly not do) in the preprocessing pipeline.


### 5.1 Near-duplicate rental events

The audit in section 4 found no exact duplicates. But sometimes a system can record the same rental twice — for example, if the user tapped the button twice, or the backend wrote one event twice. These wouldn't show up as exact duplicates because their `id` is different.

Here we look for pairs of events that might be such cases: same user, less than 5 seconds apart. If the location is the same — possibly a double-tap. If the location is different — that is physically impossible for one person, so something went wrong in the data. We run this on both rental sources (registered and direct).


In [9]:
def near_dup_check(df: pd.DataFrame, label: str) -> None:
    """Count suspicious event pairs from the same user within 5 seconds.

    Splits findings by whether the two events share a location (possible
    double-tap) or differ (physically impossible — likely data error).

    Parameters
    ----------
    df : pd.DataFrame
        Rental events with columns ``user_id``, ``datetime``, ``location_id``.
    label : str
        Label for the printout header.
    """
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], format="ISO8601")
    df = df.sort_values(["user_id", "datetime"])

    df["time_diff"] = df.groupby("user_id")["datetime"].diff()

    short_5s = df["time_diff"] < pd.Timedelta(seconds=5)
    same_loc = df["location_id"] == df.groupby("user_id")["location_id"].shift(1)

    n      = len(df)
    n_same = (short_5s &  same_loc).sum()
    n_diff = (short_5s & ~same_loc).sum()

    print(f"{label}  (of {n:,} events)")
    print(f"  same location      (potential double-tap):     {n_same:>5} ({n_same / n * 100:.4f}%)")
    print(f"  different location (physically impossible):    {n_diff:>5} ({n_diff / n * 100:.4f}%)\n")

near_dup_check(registered_rentals_data, "registered")
near_dup_check(direct_rentals_data,     "direct    ")

registered  (of 2,672,662 events)
  same location      (potential double-tap):       154 (0.0058%)
  different location (physically impossible):     3198 (0.1197%)

direct      (of 620,017 events)
  same location      (potential double-tap):        15 (0.0024%)
  different location (physically impossible):      251 (0.0405%)



### Observations

Both rental sources show the same pattern: a small number of same-location events (potential double-taps) and a slightly larger number of different-location events (physically impossible). All rates are very low — under 0.13% for registered, under 0.05% for direct.

We can't tell for sure whether any single event is a real duplicate or something legitimate — the CSV doesn't have the extra info (like a `device_id` or session ID) we would need to decide.

**We keep all rows as-is.** Removing them would risk dropping real rentals; keeping them changes hourly counts by at most a fraction of a percent. Worth a deeper look later at the system that produces the data.


### 5.2 Weather coverage of rental hours

Section 4 noted that weather is missing 165 hours inside the 2-year window. The question deferred to here is whether any of those gaps contain rental events. If yes, we would lose those rentals when joining with weather. If no, the gaps are aligned across feeds and an inner-join on the hourly timestamp is safe.


In [10]:
weather_hours = pd.Index(pd.to_datetime(weather_data["datetime"], format="ISO8601"))
expected_hours = pd.date_range(weather_hours.min(), weather_hours.max(), freq="h")
missing_weather_hours = expected_hours.difference(weather_hours)

reg_hours    = pd.to_datetime(registered_rentals_data["datetime"], format="ISO8601").dt.floor("h")
direct_hours = pd.to_datetime(direct_rentals_data["datetime"], format="ISO8601").dt.floor("h")
rental_hours = pd.Index(reg_hours).union(pd.Index(direct_hours))

gap_with_rentals = missing_weather_hours.intersection(rental_hours)

print(f"Weather is missing {len(missing_weather_hours)} hours inside the 2-year window.")
print(f"Of those, {len(gap_with_rentals)} have rental events.")

Weather is missing 165 hours inside the 2-year window.
Of those, 0 have rental events.


### Observations

All 165 missing weather hours have **zero rental events**. The gaps line up across both rental feeds (most likely planned downtime, or one incident that took both feeds down) — not a weather-only outage.

**An inner-join of rentals and weather on the hourly timestamp would be safe:** no rental data is lost, and no missing weather needs filling. (We still use a left-join in §6.3 as a defensive choice.)

### 5.3 Sensor outage: humidity = 0%

Section 4 flagged that `humidity` reaches exactly `0.0%` — physically impossible since relative humidity essentially never reaches absolute zero in nature. Here we count those rows, see when they occur, and check whether only the `humidity` column is broken or the whole row is bad.


In [11]:
w = weather_data.copy()
w["datetime"] = pd.to_datetime(w["datetime"], format="ISO8601")

bad = w[w["humidity"] == 0]
bad_dates = sorted(bad["datetime"].dt.date.unique())
cond_counts = ", ".join(f"{k}={v}" for k, v in bad["conditions"].value_counts().items())

print(f"Rows with humidity == 0: {len(bad)}")
print(f"Affected dates:          {bad_dates}")

print(f"\nOther columns on those rows look normal:")
print(f"  temperature_c: {bad.temperature_c.min()}..{bad.temperature_c.max()} °C")
print(f"  windspeed_kmh: {bad.windspeed_kmh.min()}..{bad.windspeed_kmh.max()} km/h")
print(f"  conditions:    {cond_counts}")

Rows with humidity == 0: 22
Affected dates:          [datetime.date(2011, 3, 10)]

Other columns on those rows look normal:
  temperature_c: 8.0..12.7 °C
  windspeed_kmh: 6.0..39.0 km/h
  conditions:    light_rain=20, clouds=2


### Observations

All 22 broken rows fall on a single calendar day. Only the `humidity` column is broken — temperature, wind, and conditions look reasonable for a rainy day. So this is an isolated sensor outage, not a row-wide failure.

We don't fix this in the exploration notebook. **Action required at preprocessing stage:** in the weather asset of the pipeline, replace `humidity == 0` with NaN, then either interpolate from neighboring hours or let the downstream model handle the missing values — the choice belongs to the pipeline implementation.


## 6. Building the Combined Dataset

---

With the data quality flags from sections 4–5 handled, we now build the combined dataset that will be the output of the preprocessing pipeline. The steps follow the week-2 handout:

1. Aggregate the two rental sources to hourly counts.
2. Join with weather on the hourly timestamp (inner-join is safe — verified in §5.2).
3. Add `is_holiday` from the holiday calendar.
4. Derive standard time-based features from `datetime` (year, month, day, hour, dayofweek, is_weekend).

The result is **one row per operating hour** with rental counts, weather, holiday flag, and time features — ready for downstream modeling.


### 6.1 Hourly rental aggregation

Both rental sources record one row per individual rental event. Here we aggregate them to **counts per hour per location**, producing `registered_rentals`, `direct_pickups`, and `total_rentals`.

We also complete the grid: every `(hour, location)` pair in the operational window gets a row, with zeros where no rentals happened. Without this, "quiet" pairs would be silently missing from the final dataset and the model would never learn the "no demand" pattern at low-activity kiosks. The result matches the hourly grain of the weather data.


In [12]:
rentals = pd.concat([
    registered_rentals_data.assign(is_registered=True),
    direct_rentals_data.assign(is_registered=False),
], ignore_index=True)
rentals["datetime_hourly"] = pd.to_datetime(rentals["datetime"], format="ISO8601").dt.floor("h")

reg_counts = (
    rentals[rentals["is_registered"]]
    .groupby(["datetime_hourly", "location_id"]).size()
    .reset_index(name="registered_rentals")
)
direct_counts = (
    rentals[~rentals["is_registered"]]
    .groupby(["datetime_hourly", "location_id"]).size()
    .reset_index(name="direct_pickups")
)

all_hours = sorted(rentals["datetime_hourly"].unique())
all_locations = sorted(rentals["location_id"].unique())
full_grid = pd.merge(
    pd.DataFrame({"datetime_hourly": all_hours}),
    pd.DataFrame({"location_id": all_locations}),
    how="cross",
)

hourly_rentals = (
    full_grid
    .merge(reg_counts, on=["datetime_hourly", "location_id"], how="left")
    .merge(direct_counts, on=["datetime_hourly", "location_id"], how="left")
    .fillna(0)
    .astype({"registered_rentals": int, "direct_pickups": int})
)
hourly_rentals["total_rentals"] = (
    hourly_rentals["registered_rentals"] + hourly_rentals["direct_pickups"]
)

print(f"Shape: {hourly_rentals.shape}")
print(hourly_rentals.head().to_string(index=False))
hourly_rentals.info()

Shape: (364959, 5)
datetime_hourly  location_id  registered_rentals  direct_pickups  total_rentals
     2011-01-01            0                   0               0              0
     2011-01-01            1                   0               0              0
     2011-01-01            2                   0               1              1
     2011-01-01            3                   1               0              1
     2011-01-01            4                   2               0              2
<class 'pandas.DataFrame'>
RangeIndex: 364959 entries, 0 to 364958
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   datetime_hourly     364959 non-null  datetime64[us]
 1   location_id         364959 non-null  int64         
 2   registered_rentals  364959 non-null  int64         
 3   direct_pickups      364959 non-null  int64         
 4   total_rentals       364959 non-null  int64         
dty

### Observations

- The grid is complete: 364,959 rows = 17,379 hours × 21 locations.
- Hours with no activity are now explicit zero-rows instead of being missing.
- Counts are preserved: `registered_rentals + direct_pickups` across the table matches the raw event totals (verified in §6.5).

### 6.2 Time-derived features

From the hourly timestamp, derive a small set of time features the downstream model can use to capture daily, weekly, and seasonal patterns: `month`, `hour_of_day`, `day_of_week`, `is_weekend`. We surface the raw signal here; encoding decisions (cyclic sin/cos, ordinal, one-hot) are deferred to modeling.

In [13]:
dt = hourly_rentals["datetime_hourly"]
hourly_rentals["month"]       = dt.dt.month
hourly_rentals["hour_of_day"] = dt.dt.hour
hourly_rentals["day_of_week"] = dt.dt.dayofweek
hourly_rentals["is_weekend"]  = (dt.dt.dayofweek >= 5).astype(int)

print(f"Shape: {hourly_rentals.shape}")
print(hourly_rentals.head().to_string(index=False))

Shape: (364959, 9)
datetime_hourly  location_id  registered_rentals  direct_pickups  total_rentals  month  hour_of_day  day_of_week  is_weekend
     2011-01-01            0                   0               0              0      1            0            5           1
     2011-01-01            1                   0               0              0      1            0            5           1
     2011-01-01            2                   0               1              1      1            0            5           1
     2011-01-01            3                   1               0              1      1            0            5           1
     2011-01-01            4                   2               0              2      1            0            5           1


### Observations

- All four features are integers — easy to encode later (cyclic, one-hot, or kept as-is for tree-based models).
- The full `datetime_hourly` column is kept too, so any extra feature (year, day of month, week of year) can still be derived downstream without re-joining.

### 6.3 Add weather (with humidity fix and conditions encoding)

1. Apply the fix from §5.3: replace `humidity == 0` with NaN, then linear-interpolate from neighbours.
2. Encode `conditions` as **ordinal integer** 0..3 (`clear < clouds < light_rain < heavy_rain`). Categories are genuinely ordered by precipitation severity, so an ordinal code keeps the signal in one compact column. Downstream model is free to one-hot it if needed.
3. Left-join the cleaned weather onto `hourly_rentals` by `datetime_hourly`.

Weather has one row per hour (citywide). The join broadcasts each weather row across all 21 locations sharing that hour. We use left-join defensively — every `hourly_rentals` row has a matching weather row today, but left-join makes any future mismatch visible as NaN instead of silently dropping rows.


In [14]:
WEATHER_SEVERITY = {"clear": 0, "clouds": 1, "light_rain": 2, "heavy_rain": 3}

clean_weather = weather_data.copy()
clean_weather["datetime_hourly"] = pd.to_datetime(clean_weather["datetime"], format="ISO8601").dt.floor("h")

clean_weather["humidity"] = clean_weather["humidity"].replace(0, np.nan)
clean_weather["humidity"] = clean_weather["humidity"].interpolate("linear")

# Ordinal encoding for conditions: clear < clouds < light_rain < heavy_rain.
clean_weather["conditions"] = clean_weather["conditions"].map(WEATHER_SEVERITY)

clean_weather = clean_weather.drop(columns=["datetime", "id"])

dataset = hourly_rentals.merge(clean_weather, on="datetime_hourly", how="left")

print(f"Shape: {dataset.shape}")
print(dataset.head().to_string(index=False))

Shape: (364959, 14)
datetime_hourly  location_id  registered_rentals  direct_pickups  total_rentals  month  hour_of_day  day_of_week  is_weekend  conditions  temperature_c  perceived_temperature_c  humidity  windspeed_kmh
     2011-01-01            0                   0               0              0      1            0            5           1           0            3.3                      3.0      81.0            0.0
     2011-01-01            1                   0               0              0      1            0            5           1           0            3.3                      3.0      81.0            0.0
     2011-01-01            2                   0               1              1      1            0            5           1           0            3.3                      3.0      81.0            0.0
     2011-01-01            3                   1               0              1      1            0            5           1           0            3.3                     

### Observations

- The combined table is now renamed to `dataset`, since it carries more than rentals (weather + time features, holidays added next).
- Width grows from 9 to 14 columns; row count is unchanged — the left-join produced one weather match per row, as expected from §5.2.
- `humidity == 0` rows from §5.3 are gone (interpolated), and `conditions` is now an integer 0..3 instead of a string.

### 6.4 Add holiday flag

Left-join the holiday calendar on the date portion of `datetime`. Rows whose date is in the calendar get `is_holiday = 1`; the rest get `0`. We use a left-join (not inner) because the vast majority of hours are non-holidays and must be kept.


In [15]:
holiday_dates = set(pd.to_datetime(holiday_calendar_data["date"], format="ISO8601").dt.date)
dataset["is_holiday"] = dataset["datetime_hourly"].dt.date.isin(holiday_dates).astype(int)

print(f"Shape: {dataset.shape}")
print(dataset.head().to_string(index=False))

Shape: (364959, 15)
datetime_hourly  location_id  registered_rentals  direct_pickups  total_rentals  month  hour_of_day  day_of_week  is_weekend  conditions  temperature_c  perceived_temperature_c  humidity  windspeed_kmh  is_holiday
     2011-01-01            0                   0               0              0      1            0            5           1           0            3.3                      3.0      81.0            0.0           0
     2011-01-01            1                   0               0              0      1            0            5           1           0            3.3                      3.0      81.0            0.0           0
     2011-01-01            2                   0               1              1      1            0            5           1           0            3.3                      3.0      81.0            0.0           0
     2011-01-01            3                   1               0              1      1            0            5           1

### Observations

- Final width is 15 columns; row count unchanged — the date-based lookup keeps every hour.
- `is_holiday == 1` covers roughly 21 days × 24 hours × 21 locations ≈ 10,584 rows, about 2.9% of the dataset. A rare-but-meaningful flag, as expected.

### 6.5 Final dataset

Schema, head, row count, and sanity checks: no rental-data lost across the joins, all expected columns are present with correct dtypes, no unexpected NaN.

In [16]:
dataset.info()

print()
print(dataset.head().to_string(index=False))

raw_events_total = len(registered_rentals_data) + len(direct_rentals_data)
print(f"\nsum(total_rentals): {dataset['total_rentals'].sum():,}  (expect {raw_events_total:,} = raw events)")

<class 'pandas.DataFrame'>
RangeIndex: 364959 entries, 0 to 364958
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   datetime_hourly          364959 non-null  datetime64[us]
 1   location_id              364959 non-null  int64         
 2   registered_rentals       364959 non-null  int64         
 3   direct_pickups           364959 non-null  int64         
 4   total_rentals            364959 non-null  int64         
 5   month                    364959 non-null  int32         
 6   hour_of_day              364959 non-null  int32         
 7   day_of_week              364959 non-null  int32         
 8   is_weekend               364959 non-null  int64         
 9   conditions               364959 non-null  int64         
 10  temperature_c            364959 non-null  float64       
 11  perceived_temperature_c  364959 non-null  float64       
 12  humidity                 36

### Observations

- Every column is non-null — no gaps after the joins and the humidity fix.
- Dtypes are clean: one `datetime64`, numeric features as `int` or `float`, binary flags as `int`.
- Row-count sanity check passes: `sum(total_rentals) == raw events total` — no rentals were lost in the aggregation or the joins.

The dataset is ready to be produced by the Dagster pipeline and consumed by the modeling step.

## 7. Summary

---

**Pipeline output.** One row per `(hour, location)` over the 2-year window — 364,959 rows × 15 columns. Columns: `datetime_hourly`, `location_id`, three rental counts (`registered_rentals`, `direct_pickups`, `total_rentals`), four time features (`month`, `hour_of_day`, `day_of_week`, `is_weekend`), four weather columns (`conditions` as ordinal 0..3, `temperature_c`, `perceived_temperature_c`, `humidity`, `windspeed_kmh`), and `is_holiday`.

**Key decisions from exploration:**

- Near-duplicate rental events (§5.1) — kept as-is; impact below 0.13%, and we lack the metadata to decide per case.
- Weather gaps (§5.2) — 165 missing hours contain zero rentals; the weather join loses no rental data.
- Humidity sensor outage (§5.3) — replace `humidity == 0` with NaN and linear-interpolate from neighbours.
- `conditions` encoding — ordinal 0..3 (`clear < clouds < light_rain < heavy_rain`); rare `heavy_rain` class kept as the highest level.
- Rentals × weather join — left-join defensively, even though §5.2 shows an inner-join would be safe today.
- Holidays — binary `is_holiday` flag from a date-level lookup.
- Time features — only `month`, `hour_of_day`, `day_of_week`, `is_weekend`; the raw `datetime_hourly` is kept for any extra feature later.

**Parked for later:** open items on holiday modeling, tracked separately.